In [0]:
TABLE = "`p-cdh`.derivative.dexposure"

# WIDGETS

dbutils.widgets.dropdown("mode", "manual", ["manual", "file"], "1. Πηγη παραμετρων")
dbutils.widgets.text("input_file", "/dbfs/FileStore/close_date_input.txt", "2. Αρχειο (αν mode=file)")
dbutils.widgets.text("cdi", "", "3. CDI (αν mode=manual)")
dbutils.widgets.text("name", "", "4. Name / PartyName (αν mode=manual)")
dbutils.widgets.text("tax_number", "", "5. Tax Number (προαιρετικο)")
dbutils.widgets.text("agreement_key", "", "6. Agreement Key (αν mode=manual)")

MODE = dbutils.widgets.get("mode")
INPUT_FILE = dbutils.widgets.get("input_file")


import re

# Καθαρισμος κρυφων χαρακτηρων: non-breaking space, zero-width, narrow nbsp κλπ.
def clean(s):
    if s is None:
        return ""
    s = str(s)
    for ch in ("\u00a0", "\u200b", "\u200c", "\u200d", "\ufeff", "\u202f", "\t"):
        s = s.replace(ch, " ")
    return s.strip()

PARTIES = []

if MODE == "file":
    import os

    # Αν το path ειναι φακελος -> βρες τα .txt μεσα του και διαβασε τα ολα.
    # Αν ειναι αρχειο -> διαβασε το κατευθειαν.

    if os.path.isdir(INPUT_FILE):
        txts = sorted(
            os.path.join(INPUT_FILE, fn)
            for fn in os.listdir(INPUT_FILE)
            if fn.lower().endswith(".txt")
        )
        if not txts:
            raise FileNotFoundError(
                f"Ο φακελος '{INPUT_FILE}' δεν εχει .txt αρχειο. "
                f"Δωσε το πληρες path μεχρι το αρχειο, π.χ. {INPUT_FILE}/close_date_input.txt"
            )
        parts = []
        for path in txts:
            with open(path, "r", encoding="utf-8") as f:
                parts.append(f.read())
        raw = "\n".join(parts)
        print(f"Διαβαστηκαν {len(txts)} αρχεια απο τον φακελο {INPUT_FILE}")
    else:
        with open(INPUT_FILE, "r", encoding="utf-8") as f:
            raw = f.read()

    for chunk in re.split(r"(?=\d+\s*\.\s*CDI\s*:)", raw):
        chunk = chunk.strip()
        if "CDI" not in chunk.upper():
            continue
        m_cdi = re.search(r"CDI\s*:\s*([^,\n]+)", chunk, re.IGNORECASE)
        m_name = re.search(r"NAME\s*:\s*(.+?)\s*,\s*(?:ΑΦΜ|AFM)\s*:", chunk, re.IGNORECASE | re.DOTALL)
        m_afm = re.search(r"(?:ΑΦΜ|AFM)\s*:\s*([^,\n]+)", chunk, re.IGNORECASE)
        m_agr = re.search(r"(?:Συμβ[αά]σεις|Agreements?)\s*:\s*(.+)$", chunk, re.IGNORECASE | re.DOTALL)
        if not m_cdi:
            continue
        cdi = clean(m_cdi.group(1))
        name = clean(m_name.group(1)) if m_name else ""
        tax = clean(m_afm.group(1)) if m_afm else ""
        if m_agr:
            for a in m_agr.group(1).split(","):
                if a:
                    PARTIES.append({"cdi": cdi, "name": name, "tax": tax, "agreement": a})
    print(f"File mode: {len(PARTIES)} εγγραφες απο {INPUT_FILE}")
else:
    cdi = dbutils.widgets.get("cdi").strip()
    name = dbutils.widgets.get("name").strip()
    tax = dbutils.widgets.get("tax_number").strip()
    agreement_key = dbutils.widgets.get("agreement_key").strip()
    if not cdi or not agreement_key:
        raise ValueError("Στο manual mode τα widgets 'cdi' και 'agreement_key' ειναι υποχρεωτικα.")
    PARTIES.append({"cdi": cdi, "name": name, "tax": tax, "agreement": agreement_key})
    print("Manual mode: 1 εγγραφη απο τα widgets")

# ΧΤΙΣΙΜΟ QUERY 
# Αφαιρεση διπλοτυπων (σε περιπτωση που το parsing εγινε 2 φορες)
_seen = set()
_unique = []
for p in PARTIES:
    key = (clean(p["cdi"]), clean(p["agreement"]))
    if key not in _seen:
        _seen.add(key)
        _unique.append(p)
PARTIES = _unique
print(f"Μοναδικες εγγραφες: {len(PARTIES)}")

# Ψαχνει ΜΟΝΟ με cdi + AgreementKey (ακριβως οπως το SQL που δουλευει).
# Το name/tax ΔΕΝ μπαινουν στο φιλτρο - ερχονται μονο στο output απο τη βαση.
selects = []
for p in PARTIES:
    cdi_sql = clean(p["cdi"]).replace("'", "''")
    agr_sql = clean(p["agreement"]).replace("'", "''")

selects = []
for p in PARTIES:
    cdi_sql = clean(p["cdi"]).replace("'", "''")
    agr_sql = clean(p["agreement"]).replace("'", "''")
    selects.append(f"""
    SELECT cdi,
           AgreementKey,
           PartyName,
           PartyTaxRegistrationNumber,
           close_date
    FROM (
        SELECT '{cdi_sql}'                        AS cdi,
               '{agr_sql}'                        AS AgreementKey,
               PartyName                          AS PartyName,
               PartyTaxRegistrationNumber         AS PartyTaxRegistrationNumber,
               ClosedDate                         AS close_date,
               ROW_NUMBER() OVER (
                   PARTITION BY cdi, AgreementKey
                   ORDER BY ClosedDate DESC
               )                                  AS rn
        FROM {TABLE}
        WHERE cdi = '{cdi_sql}'
          AND AgreementKey = '{agr_sql}'
    ) t
    WHERE rn = 1
    """)

full_query = "\nUNION ALL\n".join(selects)
result = spark.sql(full_query)
result.show(truncate=False)
